# Imputation

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

## Experiment

Define output of experiment:

In [2]:
OUTPUTS = PROJECT_ROOT / "outputs"
OUTPUTS.mkdir(parents=True, exist_ok=True)

Import data, split into train/test and scale on train data.

In [3]:
from src.data import load_ett, chronological_split, scale

train, test = chronological_split(load_ett())
train_s, test_s, scaler = scale(train=train, test=test)

Set location for storing reconstructed datasets.

In [4]:
from src.experiments.artifacts import ReconstructedStore

store = ReconstructedStore(Path("outputs/reconstructed"))

Initialise experiment.

In [5]:
from src.experiments.imputation import ImputationExperiment
from src.imputers import IMPUTERS

experiment = ImputationExperiment(
    mechanisms=["mar", "mcar", "mnar"],
    proportions=[10, 30, 50],
    seeds=range(10),
    imputer_names=list(IMPUTERS)
)

Run experiment, store reconstructed dataframes and save reconstruction metrics.

In [6]:
recon_df = experiment.run(train=train, store=store)
recon_df.to_parquet(OUTPUTS / "reconstruction_metrics.parquet")


## Results

In [10]:
import pandas as pd
df = pd.read_parquet(OUTPUTS / "reconstruction_metrics.parquet")

In [12]:
df.head()

,mechanism,proportion,seed,imputer,reconstruction_rmse,reconstruction_mae
0,mar,10,0,locf,0.659715,0.407732
1,mar,10,0,ks,0.659715,0.407732
2,mar,10,0,arima,0.659715,0.407732
3,mar,10,0,lstm,0.659715,0.407732
4,mar,30,0,locf,0.792416,0.477191


In [ ]:
df.groupby(["imputer", "mechanism", "proportion"]).agg(
    rmse=("reconstruction_rmse", "mean"),
    mae=("reconstruction_mae", "mean")
    )

rmse       mae
imputer mechanism proportion                    
arima   mar       10          0.668138  0.409382
                  30          0.791434  0.477369
                  50          1.126566  0.657456
        mcar      10          0.849380  0.426648
                  30          0.931026  0.468115
                  50          1.081540  0.537892
        mnar      10          0.675228  0.410417
                  30          0.786931  0.472063
                  50          1.102487  0.638355
ks      mar       10          0.668138  0.409382
                  30          0.791434  0.477369
                  50          1.126566  0.657456
        mcar      10          0.849380  0.426648
                  30          0.931026  0.468115
                  50          1.081540  0.537892
        mnar      10          0.675228  0.410417
                  30          0.786931  0.472063
                  50          1.102487  0.638355
locf    mar       10          0.668138  0.409382
                  30          0.791434  0.477369
                  50          1.126566  0.657456
        mcar      10          0.849380  0.426648
                  30          0.931026  0.468115
                  50          1.081540  0.537892
        mnar      10          0.675228  0.410417
                  30          0.786931  0.472063
                  50          1.102487  0.638355
lstm    mar       10          0.668138  0.409382
                  30          0.791434  0.477369
                  50          1.126566  0.657456
        mcar      10          0.849380  0.426648
                  30          0.931026  0.468115
                  50          1.081540  0.537892
        mnar      10          0.675228  0.410417
                  30          0.786931  0.472063
                  50          1.102487  0.638355